# Práctica · Verificación de firmas manuscritas offline
**Procesamiento de Imágenes, Audio y Vídeo** · Grado en Ciencia e Ingeniería de Datos

Autores: _Nombre Apellido_ y _Nombre Apellido_

En este notebook construimos un verificador de firmas **paramétrico** con OpenCV. Lo hacemos paso a paso:

1. Cargar las firmas
2. Preprocesar las imágenes
3. Extraer las características
4. Poner las características en la misma escala
5. Crear el patrón de cada usuario y puntuar las firmas
6. Medir los errores (FAR, FRR, EER y curva DET)
7. Fijar el umbral con los usuarios de desarrollo
8. Evaluar con los usuarios de evaluación

## Hipótesis

**Planteamiento.** Primero limpiamos cada firma (filtro de mediana, binarización con Otsu y recorte) y después sacamos 5 medidas sencillas: la densidad de tinta, la proporción entre ancho y alto, la posición del centro de la firma en horizontal y en vertical, y el número de trazos separados (componentes conexas). Con las 10 firmas de referencia calculamos la firma "media" de cada persona. Para comparar una firma nueva, ponemos todas las medidas en la misma escala y calculamos lo lejos que está de esa firma media: cuanto más cerca, más se parece.

**Umbral.** Usaremos un único umbral global, el mismo para todos los usuarios. Lo fijaremos con los 30 usuarios de desarrollo en el punto en el que la FAR y la FRR son iguales (el punto del EER). Para encontrarlo, puntuaremos todas las firmas de prueba de los usuarios de desarrollo, calcularemos la FAR y la FRR para cada umbral posible y nos quedaremos con el valor en el que ambas coinciden. Después no lo tocaremos, ni en evaluación ni con los usuarios nuevos de la prueba final.

**Qué esperamos.**
- Que funcione, pero con bastante error: un **EER de entre el 20 % y el 30 %**.
- Que le cueste distinguir las **falsificaciones hábiles**. Quien falsifica imita bien el aspecto general de la firma, y ese aspecto general es lo único que miden nuestras características.
- Que, al usar el mismo umbral para todos, la **FAR y la FRR salgan algo descompensadas** en evaluación, ya que no todo el mundo firma igual de regular.

## 1. Cargar las firmas

Usamos la base de datos CEDAR del Campus Virtual. Cada usuario tiene 24 firmas genuinas y 24 falsificaciones:

| Firmas | Para qué las usamos |
|---|---|
| Genuinas 1–10 | Referencias: con ellas creamos el patrón del usuario |
| Genuinas 11–24 | Pruebas: el sistema debería aceptarlas (sirven para la FRR) |
| Falsificaciones 1–24 | Pruebas: el sistema debería rechazarlas (sirven para la FAR) |

Los 30 primeros usuarios son de **desarrollo** (con ellos ajustamos todo) y el resto son de **evaluación** (solo medimos, no se ajusta nada).

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
# Carpetas donde están las firmas
CARPETA_GENUINAS = "data/genuinas/full_org/full_org"
CARPETA_FALSAS = "data/falsificadas/full_forg/full_forg"

# Las genuinas 1 a 10 son las referencias
N_REFERENCIAS = 10


def ruta_firma(usuario, numero, tipo):
    """Devuelve la ruta de una firma. tipo es "genuina" o "falsificada"."""
    if tipo == "genuina":
        return os.path.join(CARPETA_GENUINAS, f"original_{usuario}_{numero}.png")
    else:
        return os.path.join(CARPETA_FALSAS, f"forgeries_{usuario}_{numero}.png")

In [ ]:
# Sacamos los números de usuario de los nombres de archivo (original_<usuario>_<numero>.png)
usuarios = set()
for nombre in os.listdir(CARPETA_GENUINAS):
    if nombre.endswith(".png"):
        partes = nombre[:-4].split("_")
        usuarios.add(int(partes[1]))
usuarios = sorted(usuarios)

usuarios_desarrollo = usuarios[:30]
usuarios_evaluacion = usuarios[30:]

print("Usuarios en total:", len(usuarios))
print("Desarrollo:", usuarios_desarrollo)
print("Evaluación:", usuarios_evaluacion)

In [ ]:
# Vemos algunas firmas de un usuario
usuario = usuarios[0]

fig, ejes = plt.subplots(2, 4, figsize=(14, 5))
for i in range(4):
    genuina = cv2.imread(ruta_firma(usuario, i + 1, "genuina"), cv2.IMREAD_GRAYSCALE)
    falsa = cv2.imread(ruta_firma(usuario, i + 1, "falsificada"), cv2.IMREAD_GRAYSCALE)
    ejes[0, i].imshow(genuina, cmap="gray")
    ejes[0, i].set_title(f"Genuina {i + 1}")
    ejes[1, i].imshow(falsa, cmap="gray")
    ejes[1, i].set_title(f"Falsificación {i + 1}")
for eje in ejes.flat:
    eje.axis("off")
plt.suptitle(f"Usuario {usuario}")
plt.show()

## 2. Preprocesado

Antes de medir nada, limpiamos cada imagen:

1. **Leer en escala de grises.**
2. **Filtro de mediana 3×3**: quita los puntitos de ruido del escaneo sin difuminar el trazo.
3. **Binarización con Otsu**: OpenCV calcula solo el umbral que separa la tinta del fondo. Usamos la versión invertida para que la tinta quede en blanco (255) y el fondo en negro (0).
4. **Recorte**: nos quedamos con el rectángulo que contiene la tinta, para que no influya en qué parte del papel estaba la firma.

In [ ]:
def preprocesar(ruta):
    gris = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
    gris = cv2.medianBlur(gris, 3)
    _, binaria = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Recorte: buscamos las filas y columnas donde hay tinta
    filas, columnas = np.where(binaria > 0)
    recortada = binaria[filas.min():filas.max() + 1, columnas.min():columnas.max() + 1]
    return recortada

In [ ]:
# Comparamos una firma antes y después del preprocesado
ruta = ruta_firma(usuarios[0], 1, "genuina")

fig, ejes = plt.subplots(1, 2, figsize=(12, 3))
ejes[0].imshow(cv2.imread(ruta, cv2.IMREAD_GRAYSCALE), cmap="gray")
ejes[0].set_title("Original")
ejes[1].imshow(preprocesar(ruta), cmap="gray")
ejes[1].set_title("Mediana + Otsu + recorte")
for eje in ejes:
    eje.axis("off")
plt.show()

## 3. Características

De cada firma preprocesada sacamos 5 medidas:

| # | Característica | Cómo se calcula |
|---|---|---|
| 1 | Densidad de tinta | píxeles de tinta / (ancho × alto) |
| 2 | Relación ancho/alto | ancho / alto |
| 3 | Centro x | media de las columnas con tinta / ancho |
| 4 | Centro y | media de las filas con tinta / alto |
| 5 | Componentes conexas | número de trazos separados (sin contar manchas de menos de 10 píxeles, que son ruido) |

El centro se divide entre el ancho y el alto para que no dependa del tamaño de la firma.

In [ ]:
NOMBRES = ["densidad", "ancho_alto", "centro_x", "centro_y", "componentes"]


def caracteristicas(firma):
    alto, ancho = firma.shape
    filas, columnas = np.where(firma > 0)   # posiciones de los píxeles de tinta

    # 1. Densidad de tinta
    densidad = len(filas) / (alto * ancho)

    # 2. Relación ancho/alto
    ancho_alto = ancho / alto

    # 3 y 4. Centro de la tinta, dividido entre el ancho y el alto
    centro_x = columnas.mean() / ancho
    centro_y = filas.mean() / alto

    # 5. Componentes conexas (la fila 0 de estadisticas es el fondo, así que la quitamos)
    _, _, estadisticas, _ = cv2.connectedComponentsWithStats(firma)
    areas = estadisticas[1:, cv2.CC_STAT_AREA]
    componentes = np.sum(areas >= 10)

    return [densidad, ancho_alto, centro_x, centro_y, componentes]

In [ ]:
# Probamos con una firma
valores = caracteristicas(preprocesar(ruta_firma(usuarios[0], 1, "genuina")))
for nombre, valor in zip(NOMBRES, valores):
    print(f"{nombre:12s} {valor:.3f}")

Ahora sacamos las características de **todas** las firmas y las guardamos en una tabla con una fila por firma. Tarda un poco.

In [ ]:
filas_tabla = []
for usuario in usuarios:
    for numero in range(1, 25):
        for tipo in ["genuina", "falsificada"]:
            firma = preprocesar(ruta_firma(usuario, numero, tipo))
            filas_tabla.append([usuario, numero, tipo] + caracteristicas(firma))

datos = pd.DataFrame(filas_tabla, columns=["usuario", "firma", "tipo"] + NOMBRES)
print("Firmas procesadas:", len(datos))
datos.head()

## 4. Poner las características en la misma escala

Cada característica tiene una escala muy distinta: la densidad ronda 0,1 y las componentes conexas pueden valer 10 o más. Si las usáramos tal cual, las de números grandes mandarían en la distancia.

Por eso las **estandarizamos**: a cada una le restamos su media y la dividimos entre su desviación típica. La media y la desviación se calculan **solo con los usuarios de desarrollo**.

In [ ]:
desarrollo = datos[datos.usuario.isin(usuarios_desarrollo)]
medias = desarrollo[NOMBRES].mean()
desviaciones = desarrollo[NOMBRES].std()

datos_esc = datos.copy()
datos_esc[NOMBRES] = (datos[NOMBRES] - medias) / desviaciones
datos_esc.head()

## 5. Patrón del usuario y puntuación

- **Patrón**: la media de las características de sus 10 firmas de referencia, es decir, su "firma media".
- **Puntuación** de una firma: la distancia euclídea entre la firma y el patrón, con el signo cambiado. Así, cuanto mayor es la puntuación, más se parece (la máxima posible es 0).

Se puntúan las firmas de prueba: genuinas 11–24 y las 24 falsificaciones.

In [ ]:
def puntuar(datos_esc, lista_usuarios):
    resultados = []
    for usuario in lista_usuarios:
        firmas_usuario = datos_esc[datos_esc.usuario == usuario]

        # Patrón: media de las 10 firmas de referencia
        es_referencia = (firmas_usuario.tipo == "genuina") & (firmas_usuario.firma <= N_REFERENCIAS)
        patron = firmas_usuario[es_referencia][NOMBRES].mean().values

        # Firmas de prueba: todas las que no son de referencia
        pruebas = firmas_usuario[~es_referencia]
        for _, fila in pruebas.iterrows():
            valores = fila[NOMBRES].values.astype(float)
            distancia = np.linalg.norm(valores - patron)
            resultados.append([usuario, fila.firma, fila.tipo, -distancia])

    return pd.DataFrame(resultados, columns=["usuario", "firma", "tipo", "puntuacion"])

## 6. Medidas de error

Para un umbral dado, aceptamos las firmas con puntuación **mayor o igual** que el umbral y rechazamos el resto:

- **FAR**: % de falsificaciones aceptadas.
- **FRR**: % de genuinas rechazadas.
- **EER**: el error en el punto en que FAR = FRR. Para encontrarlo probamos como umbral cada una de las puntuaciones obtenidas.
- **Curva DET**: FRR frente a FAR para todos los umbrales, con los ejes en escala normal.

In [ ]:
def far_frr(puntuaciones, umbral):
    genuinas = puntuaciones[puntuaciones.tipo == "genuina"].puntuacion
    falsas = puntuaciones[puntuaciones.tipo == "falsificada"].puntuacion
    far = np.mean(falsas >= umbral)    # falsificaciones aceptadas
    frr = np.mean(genuinas < umbral)   # genuinas rechazadas
    return far, frr


def calcular_eer(puntuaciones):
    mejor_diferencia = 1.0
    for umbral in np.sort(puntuaciones.puntuacion.unique()):
        far, frr = far_frr(puntuaciones, umbral)
        if abs(far - frr) < mejor_diferencia:
            mejor_diferencia = abs(far - frr)
            eer = (far + frr) / 2
            umbral_eer = umbral
    return eer, umbral_eer


def dibujar_det(puntuaciones, titulo, umbral_fijo=None):
    lista_far, lista_frr = [], []
    for umbral in np.sort(puntuaciones.puntuacion.unique()):
        far, frr = far_frr(puntuaciones, umbral)
        lista_far.append(far)
        lista_frr.append(frr)

    # Escala normal: norm.ppf no admite 0 ni 1, así que recortamos un poco
    far = np.clip(lista_far, 0.001, 0.999)
    frr = np.clip(lista_frr, 0.001, 0.999)

    plt.figure(figsize=(6, 6))
    plt.plot(norm.ppf(far), norm.ppf(frr), lw=2, label="Sistema paramétrico")
    if umbral_fijo is not None:
        far_u, frr_u = far_frr(puntuaciones, umbral_fijo)
        plt.plot(norm.ppf(max(far_u, 0.001)), norm.ppf(max(frr_u, 0.001)), "ro", label="Umbral fijado")

    marcas = [0.01, 0.02, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8]
    etiquetas = [f"{100 * m:g}" for m in marcas]
    plt.xticks(norm.ppf(marcas), etiquetas)
    plt.yticks(norm.ppf(marcas), etiquetas)
    plt.xlabel("FAR (%)")
    plt.ylabel("FRR (%)")
    plt.title(titulo)
    plt.grid(alpha=0.3)
    plt.legend()

## 7. Desarrollo: fijamos el umbral

Puntuamos las firmas de los 30 usuarios de desarrollo y fijamos el umbral en el punto del EER. **Este umbral ya no se vuelve a cambiar.**

In [ ]:
punt_desarrollo = puntuar(datos_esc, usuarios_desarrollo)
eer_desarrollo, UMBRAL = calcular_eer(punt_desarrollo)

print(f"EER en desarrollo: {100 * eer_desarrollo:.2f} %")
print(f"Umbral fijado: {UMBRAL:.4f}")

In [ ]:
# Cómo se reparten las puntuaciones de las genuinas y de las falsificaciones
genuinas = punt_desarrollo[punt_desarrollo.tipo == "genuina"].puntuacion
falsas = punt_desarrollo[punt_desarrollo.tipo == "falsificada"].puntuacion

plt.figure(figsize=(8, 4))
plt.hist(genuinas, bins=40, alpha=0.6, density=True, label="Genuinas")
plt.hist(falsas, bins=40, alpha=0.6, density=True, label="Falsificaciones")
plt.axvline(UMBRAL, color="k", linestyle="--", label="Umbral")
plt.xlabel("Puntuación")
plt.title("Puntuaciones en desarrollo")
plt.legend()
plt.show()

## 8. Evaluación con el umbral fijo

Ahora usamos los usuarios de evaluación sin cambiar nada: la misma escala y el mismo umbral que en desarrollo.

In [ ]:
punt_evaluacion = puntuar(datos_esc, usuarios_evaluacion)

far, frr = far_frr(punt_evaluacion, UMBRAL)
eer_evaluacion, _ = calcular_eer(punt_evaluacion)

print(f"EER en evaluación: {100 * eer_evaluacion:.2f} %")
print(f"Con el umbral fijado ({UMBRAL:.4f}): FAR = {100 * far:.2f} %, FRR = {100 * frr:.2f} %")

In [ ]:
dibujar_det(punt_evaluacion, "Curva DET · usuarios de evaluación", UMBRAL)
plt.show()

## Conclusiones

_(Completar después de ejecutar)_

- ¿Se cumple la hipótesis? ¿El EER está entre el 20 % y el 30 %?
- Con el umbral fijado en desarrollo, ¿la FAR y la FRR salen parecidas o descompensadas?
- ¿Qué error es más importante? En una aplicación de seguridad (un banco, por ejemplo) es más grave la FAR, porque aceptar una falsificación es un fraude. En una aplicación donde prima la comodidad importa más la FRR, porque rechazar al usuario auténtico es molesto.

## Referencias y uso de IA

- Base de datos CEDAR de firmas manuscritas (Center of Excellence for Document Analysis and Recognition, Univ. at Buffalo).
- Bibliotecas: OpenCV, NumPy, pandas, SciPy y Matplotlib.

**Uso de IA:** hemos usado un asistente de IA (Claude, de Anthropic) como apoyo. _(Explicar aquí para qué lo habéis usado.)_